# ARXIV CRAWLER

Nhận vào input là ids.txt, range id [start, end), giả sử muốn chạy từ dòng đầu tiên tới dòng 2000, thì đặt start_idx = 0, end_idx = 2000

Retry HTTP 429/5xx sau khi hết mỗi lượt, không giới hạn số vòng. Archive và metadata tải vẫn được giữ; source giải nén nằm trong `<publication-key>/v<n>/`, kèm `status.json`. `done.json` chỉ có khi tất cả version của Publication trong file ID đã xử lý xong (kể cả version không có TeX). Notebook không tự bỏ qua bài dựa trên các file trạng thái này.

## Đường dẫn trên Kaggle

Chỉ cần sửa `IDS_TXT_DIR` trong ô cấu hình ngay bên dưới thành đường dẫn file `arxiv_id.txt` của dataset đã thêm vào notebook. Mỗi dòng là một ID có version, ví dụ `2409.18594v1`.

`OUTPUT_ROOT` mặc định là `/kaggle/working/arxiv-crawler`; archive, source giải nén, `status.json` và `done.json` nằm trong `OUTPUT_ROOT/output/`, còn metadata tải nằm tại `OUTPUT_ROOT/metadata.json`. Muốn đổi nơi lưu, chỉ sửa `OUTPUT_ROOT`.

Giữ `START_IDX`, `END_IDX` để chọn các dòng cần chạy; `END_IDX` không bao gồm dòng cuối này. Khi chạy local từ root repo, dùng hai đường dẫn local được chú thích sẵn trong ô cấu hình. Sau khi chỉnh cấu hình, chạy lại các ô phía dưới theo thứ tự.


In [1]:
from pathlib import Path

# Kaggle: thay bằng đường dẫn file ID trong mục Input của notebook.
IDS_TXT_DIR = Path("/kaggle/input/TEN_DATASET/arxiv_id.txt")
OUTPUT_ROOT = Path("/kaggle/working/arxiv-crawler")

# Chạy local từ root repo: dùng hai dòng dưới thay cho hai dòng trên.
# IDS_TXT_DIR = Path("src/data/arxiv_id.txt")
# OUTPUT_ROOT = Path("src/data")

SAVE_DIR = OUTPUT_ROOT / "output"
METADATA_OUTPUT_DIR = OUTPUT_ROOT / "metadata.json"
START_IDX = 0
END_IDX = 10
RETRY_ROUND_DELAY = 1


In [2]:
import arxiv
import requests
import shutil
import time
import urllib.request
from pathlib import Path

client = arxiv.Client(delay_seconds=3.0)


def get_source(arxiv_id, save_dir=SAVE_DIR):
    destination = Path(save_dir) / f"{arxiv_id}.tar.gz"
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        paper = next(client.results(arxiv.Search(id_list=[arxiv_id])))
        print(paper.title, paper.get_short_id())
        url = f"https://export.arxiv.org/src/{arxiv_id}"
        with urllib.request.urlopen(url, timeout=60) as response:
            with destination.open("wb") as output:
                shutil.copyfileobj(response, output)
        return 200
    finally:
        time.sleep(2)

/home/nhminh/miniconda3/envs/DL_Env/lib/python3.12/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (2.8.0) or chardet (7.4.3)/charset_normalizer (3.4.9) doesn't match a supported version!
  warnings.warn(


In [3]:
import json
from pathlib import Path


def dump_to_json(data: list[dict]):
    if not data:
        return
    path = Path(METADATA_OUTPUT_DIR)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        with path.open("r", encoding="utf-8") as file:
            metadata = json.load(file)
    else:
        metadata = []
    metadata.extend(data)
    with path.open("w", encoding="utf-8") as file:
        json.dump(metadata, file, ensure_ascii=False, indent=2)


def version_output(arxiv_id):
    """Tách ID có version, giữ nguyên ID gốc trong metadata."""
    base_id, version = arxiv_id.rsplit("v", 1)
    publication_key = base_id.replace("/", "-").replace(".", "-")
    return base_id, int(version), Path(SAVE_DIR) / publication_key / f"v{int(version)}"


def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(".json.tmp")
    temporary_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary_path.replace(path)


def write_done_files():
    """Chỉ đánh dấu hoàn tất khi đủ mọi version trong file ID, kể cả ngoài lát cắt chạy."""
    publications = {}
    for line in Path(IDS_TXT_DIR).read_text(encoding="utf-8").splitlines():
        if line.strip():
            base_id, version, version_dir = version_output(line.strip())
            publications.setdefault(base_id, {})[version] = version_dir

    for base_id, version_dirs in publications.items():
        paper_dir = next(iter(version_dirs.values())).parent
        done_path = paper_dir / "done.json"
        records = []
        for version_dir in version_dirs.values():
            status_path = version_dir / "status.json"
            if not status_path.exists():
                break
            record = json.loads(status_path.read_text(encoding="utf-8"))
            if record["status"] not in {"ok", "empty"}:
                break
            records.append(record)
        if len(records) != len(version_dirs):
            done_path.unlink(missing_ok=True)
            continue
        write_json(done_path, {
            "id": base_id,
            "status": "done",
            "n_versions": len(records),
            "versions_with_tex": sum(record["status"] == "ok" for record in records),
            "bytes_downloaded": sum(record["bytes_downloaded"] for record in records),
            "bytes_before": sum(record["bytes_before"] for record in records),
            "bytes_after": sum(record["bytes_after"] for record in records),
            "seconds": round(sum(record["seconds"] for record in records), 3),
        })


In [4]:
def get_id_list():
    id_list = []
    with Path(IDS_TXT_DIR).open("r", encoding="utf-8") as file:
        for index, line in enumerate(file):
            if index >= END_IDX:
                break
            if index >= START_IDX and line.strip():
                id_list.append(line.strip())
    return id_list


id_list = get_id_list()

In [5]:
def crawl_sources():
    """Giữ một lần thử mỗi ID trong mỗi lượt; chỉ retry HTTP 429/5xx ở lượt sau."""
    count = 0
    metadata_list = []
    pending_ids = id_list
    retry_round = 0
    try:
        while pending_ids:
            retry_ids = []
            for arxiv_id in pending_ids:
                count += 1
                base_id, version, version_dir = version_output(arxiv_id)
                (version_dir.parent / "done.json").unlink(missing_ok=True)
                archive_path = Path(SAVE_DIR) / f"{arxiv_id}.tar.gz"
                started = time.perf_counter()
                record = {
                    "version": version, "kind": "download_error", "status": "error", "files": [],
                    "bytes_downloaded": 0, "bytes_before": 0, "bytes_after": 0,
                }
                try:
                    res = get_source(arxiv_id=arxiv_id, save_dir=SAVE_DIR)
                    state = "success" if res == 200 else "fail"
                    if state == "success":
                        record.update(kind="unknown", status="downloaded",
                                      bytes_downloaded=archive_path.stat().st_size)
                except (OSError, StopIteration, arxiv.ArxivError, requests.exceptions.RequestException) as error:
                    print(f"Không tải được {arxiv_id}: {type(error).__name__}")
                    state = "fail"
                    record["error"] = type(error).__name__
                    http_status = getattr(error, "code", getattr(error, "status", None))
                    if isinstance(error, requests.exceptions.HTTPError) and error.response is not None:
                        http_status = error.response.status_code
                    if http_status == 429 or (http_status is not None and 500 <= http_status < 600):
                        retry_ids.append(arxiv_id)
                        print(f"[RETRY-QUEUED] {arxiv_id}: thử lại sau khi kết thúc lượt hiện tại")
                record["seconds"] = round(time.perf_counter() - started, 3)
                write_json(version_dir / "status.json", record)
                metadata_list.append({"id": arxiv_id, "path": str(archive_path), "state": state})
                if count % 100 == 0:
                    dump_to_json(metadata_list)
                    metadata_list.clear()
            dump_to_json(metadata_list)
            metadata_list.clear()
            pending_ids = retry_ids
            if pending_ids:
                retry_round += 1
                print(f"[RETRY] round={retry_round}; pending={len(pending_ids)}; wait={RETRY_ROUND_DELAY}s")
                time.sleep(RETRY_ROUND_DELAY)
    finally:
        dump_to_json(metadata_list)


crawl_sources()


"Oh LLM, I'm Asking Thee, Please Give Me a Decision Tree": Zero-Shot Decision Tree Induction and Embedding with Large Language Models 2409.18594v1
"Oh LLM, I'm Asking Thee, Please Give Me a Decision Tree": Zero-Shot Decision Tree Induction and Embedding with Large Language Models 2409.18594v2
Dynamic Competition for Attention 2409.18595v1
Dynamic Competition for Attention 2409.18595v2
ASAG2024: A Combined Benchmark for Short Answer Grading 2409.18596v1
TemporalPaD: a reinforcement-learning framework for temporal feature representation and dimension reduction 2409.18597v1
Không tải được 2409.18597v1: HTTPError
Spectral extremal problems on outerplanar and planar graphs 2409.18598v1
Spectral extremal problems on outerplanar and planar graphs 2409.18598v2
Spectral extremal problems on outerplanar and planar graphs 2409.18598v3
Deformation maps in proto-twilled Leibniz algebras 2409.18599v1


In [6]:
import gzip
import shutil
import tarfile
import tempfile
import zlib
from pathlib import Path

skipped_ids = []

for archive_path in sorted(Path(SAVE_DIR).glob("*.tar.gz")):
    arxiv_id = archive_path.name.removesuffix(".tar.gz")
    base_id, version, output_dir = version_output(arxiv_id)
    status_path = output_dir / "status.json"
    previous = json.loads(status_path.read_text(encoding="utf-8")) if status_path.exists() else {}
    # Không giải nén archive cũ hoặc dở dang của lần tải vừa thất bại.
    if previous.get("status") == "error" and previous.get("kind") == "download_error":
        continue
    (output_dir.parent / "done.json").unlink(missing_ok=True)
    started = time.perf_counter()
    record = {
        "version": version, "kind": "other", "status": "empty", "files": [],
        "bytes_downloaded": archive_path.stat().st_size,
        "bytes_before": archive_path.stat().st_size, "bytes_after": 0,
    }
    temp_path = None
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        with archive_path.open("rb") as source:
            signature = source.read(5)
        if signature.startswith(b"%PDF-"):
            record.update(kind="pdf", bytes_before=archive_path.stat().st_size)
            print(f"Bỏ qua {archive_path.name}: chỉ có PDF, không có nguồn LaTeX")
            skipped_ids.append(arxiv_id)
            continue

        if tarfile.is_tarfile(archive_path):
            record["kind"] = "tar"
            with tarfile.open(archive_path, "r:*") as archive:
                record["bytes_before"] = sum(member.size for member in archive.getmembers() if member.isfile())
                members = [
                    member for member in archive
                    if member.isfile() and Path(member.name).suffix.lower() in {".tex", ".bib"}
                ]
                # Chặn đường dẫn trong archive ghi ra ngoài thư mục đích.
                archive.extractall(output_dir, members=members, filter="data")
            kept_count = len(members)
            record["files"] = [str(Path(member.name.lstrip("/"))) for member in members]
        elif signature.startswith(b"\x1f\x8b"):
            with gzip.open(archive_path, "rb") as source:
                prefix = source.read(16384)
                if prefix.startswith(b"%PDF-"):
                    record["kind"] = "pdf"
                    record["bytes_before"] = len(prefix)
                    for chunk in iter(lambda: source.read(1024 * 1024), b""):
                        record["bytes_before"] += len(chunk)
                    print(f"Bỏ qua {archive_path.name}: GZIP chứa PDF")
                    skipped_ids.append(arxiv_id)
                    continue
                if not any(marker in prefix for marker in (
                    b"\\documentclass", b"\\documentstyle", b"\\begin{document}", b"\\input"
                )):
                    record["bytes_before"] = len(prefix)
                    for chunk in iter(lambda: source.read(1024 * 1024), b""):
                        record["bytes_before"] += len(chunk)
                    print(f"Bỏ qua {archive_path.name}: GZIP không nhận diện được nguồn LaTeX")
                    skipped_ids.append(arxiv_id)
                    continue
                with tempfile.NamedTemporaryFile(dir=output_dir, delete=False) as output:
                    temp_path = Path(output.name)
                    output.write(prefix)
                    shutil.copyfileobj(source, output)
            temp_path.replace(output_dir / "main.tex")
            kept_count = 1
            record.update(
                kind="gz_single", files=["main.tex"],
                bytes_before=(output_dir / "main.tex").stat().st_size,
            )
        else:
            print(f"Bỏ qua {archive_path.name}: định dạng không hỗ trợ")
            skipped_ids.append(arxiv_id)
            continue

        for file_path in output_dir.rglob("*"):
            if (
                (file_path.is_file() or file_path.is_symlink())
                and file_path.suffix.lower() not in {".tex", ".bib"}
                and file_path != status_path
            ):
                file_path.unlink()
        record["bytes_after"] = sum((output_dir / name).stat().st_size for name in record["files"])
        record["status"] = "ok" if record["files"] else "empty"
        print(f"Đã giải nén {archive_path.name}: giữ {kept_count} file .tex/.bib")
    except (tarfile.TarError, OSError, EOFError, zlib.error) as error:
        record.update(status="error", error=type(error).__name__, files=[])
        skipped_ids.append(arxiv_id)
        print(f"Không giải nén được {archive_path.name}: {error}")
    finally:
        if temp_path is not None:
            temp_path.unlink(missing_ok=True)
        download_seconds = previous.get("seconds", 0) if previous.get("status") == "downloaded" else 0
        record["seconds"] = round(download_seconds + time.perf_counter() - started, 3)
        write_json(status_path, record)

pass_path = Path(SAVE_DIR) / "pass.txt"
pass_path.parent.mkdir(parents=True, exist_ok=True)
pass_path.write_text("".join(f"{arxiv_id}\n" for arxiv_id in skipped_ids), encoding="utf-8")

write_done_files()


Đã giải nén 2409.18594v1.tar.gz: giữ 2 file .tex/.bib
Đã giải nén 2409.18594v2.tar.gz: giữ 1 file .tex/.bib
Đã giải nén 2409.18595v1.tar.gz: giữ 1 file .tex/.bib
Đã giải nén 2409.18595v2.tar.gz: giữ 1 file .tex/.bib
Đã giải nén 2409.18596v1.tar.gz: giữ 2 file .tex/.bib
Đã giải nén 2409.18598v1.tar.gz: giữ 1 file .tex/.bib
Đã giải nén 2409.18598v2.tar.gz: giữ 1 file .tex/.bib
Đã giải nén 2409.18598v3.tar.gz: giữ 1 file .tex/.bib
Đã giải nén 2409.18599v1.tar.gz: giữ 1 file .tex/.bib


0